# MongoDB, explained through this project

A guided tour of what the incident coding app stores in MongoDB Atlas, why it's
shaped that way, and how to query it — written for someone who has never used
Mongo.

Every cell here is **read-only**. Nothing in this notebook changes your data.
The one destructive example is commented out and marked.

---

## 1. What MongoDB is, in one table

| Spreadsheet | SQL database | MongoDB |
|---|---|---|
| workbook | database | **database** (`incidents`) |
| sheet | table | **collection** (`incidents`) |
| row | row | **document** |
| column | column | **field** |

One difference matters more than all the others: **a document is a nested JSON
object, not a flat row.** A single document can contain arrays and sub-objects
many levels deep. In a spreadsheet, "one incident cited by 3 articles, each coded
by 2 coders, each with 4 lists of characteristics" needs four sheets and a pile
of ID columns to join them. In Mongo it's *one document* that looks exactly like
the thing it describes.

That's the whole reason this project uses it.

The second difference: **there is no fixed schema.** Mongo will accept any shape
you send. That's freedom you don't want in a research dataset, so this project
adds a *validator* back on top — see §7.

> **Names are confusing here:** the Atlas *database* is called `incidents` and so
> is the *collection* inside it. Hence `db["incidents"]` — database `db`,
> collection `incidents`.

## 2. Connecting

`pymongo` is the official Python driver. Three steps: build a connection string,
make a `MongoClient`, pick a database.

The connection string (`MONGO_URI`) contains the password, so it lives in `.env`,
which is git-ignored. **Never paste a password into a notebook** — notebooks get
committed, and a credential in git history is a credential you have to rotate.

In [ ]:
from pathlib import Path
import os
from pymongo import MongoClient

# This notebook lives in docs/, so .env is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
for line in (ROOT / ".env").read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

client = MongoClient(os.environ["MONGO_URI"], serverSelectionTimeoutMS=5000)
client.admin.command("ping")          # fails loudly if the URI or network is wrong
db = client[os.environ.get("MONGO_DB", "incidents")]
coll = db["incidents"]                # the collection everything lives in
print(f"connected — database {db.name!r}, {coll.count_documents({})} incident document(s)")

## 3. The shape: one document per incident

The central design decision. An **incident** (e.g. "CNET's AI articles had
errors") is the unit of research, and several articles can report the same
incident. So the collection stores one document per incident, with everything
about it nested inside:

```
{
  _id:              ObjectId(...),        # Mongo's automatic primary key
  incident_id:      "INC-004",            # OUR key — unique, human-readable
  incident_title:   "ChatGPT is hallucinating fake links",

  documents: [                            # which articles report this incident
    { doc_id: "R95RRQFZ", url: "https://…", title: "…" },
  ],

  by_document: {                          # the coding, per article…
    "R95RRQFZ": {
      by_coder: {                         # …and per coder
        "coder1": {
          fields: { incident_aftermath: { answer: "…", comments: "…" }, … },
          quotes: [ { text, start, end, role, value }, … ],
          roles:  { actor: [...], harm: [...], factor: [...], harmed_party: [...] },
          updated_at: 2026-08-02T…
        },
        "coder2": { … }                   # same article, independent reading
      }
    }
  },

  groups_by_coder: {                      # drag-to-group claims, per coder
    "coder1": [ { id: "1", members: [ { role: "actor", value: "…" }, … ] } ]
  },
  created_at, updated_at
}
```

Read the nesting as a sentence: **incident → article → coder → their coding.**

Two things are *shared* (one copy per incident, true for everyone): `documents`
and `incident_id`/`incident_title` — which articles belong to this incident.
Everything under `by_coder` is one person's private judgement. That split is the
entire multi-coder design.

### `_id` vs `incident_id`

Every Mongo document gets an `_id` automatically. We ignore it and key on our own
`incident_id` instead, because `INC-004` means something to a human and is stable
if a document is ever re-created. A unique index enforces it (§7).

In [ ]:
# Walk into one document layer by layer — the fastest way to feel the nesting.
inc = coll.find_one()                       # find_one() = "any one document"

print("top-level fields:", sorted(inc))
print("\nincident_id:", inc["incident_id"], "|", inc.get("incident_title"))
print("\narticles in this incident:")
for d in inc.get("documents", []):
    print("   ", d["doc_id"], "—", (d.get("title") or "")[:60])

for doc_key, entry in (inc.get("by_document") or {}).items():
    coders = entry.get("by_coder", {"(legacy flat coding)": entry})
    print(f"\nby_document['{doc_key}'] coded by: {sorted(coders)}")
    for coder, coding in coders.items():
        print(f"   {coder}: {len(coding.get('quotes') or [])} quotes, "
              f"roles -> { {r: v for r, v in (coding.get('roles') or {}).items() if v} }")

## 4. Why keys-as-data (and when that's a bad idea)

`by_document` and `by_coder` use **data as field names** — the Zotero key and the
coder's name *are* the keys. The alternative would be arrays:

```js
by_document: { "R95RRQFZ": {...} }        // what we do
codings: [ { doc_id: "R95RRQFZ", ... } ]  // the array alternative
```

**Why keys win here:** updating one coder's reading of one article is a single
precise write — `by_document.R95RRQFZ.by_coder.coder1` — with no risk of touching
a neighbour. Uniqueness is free: an object can't have two identical keys. With an
array you'd need `$elemMatch`/`arrayFilters` gymnastics and could accidentally
insert duplicates.

**What it costs:** you can't index inside a dynamic key, and querying "every
document coded by coder2" means scanning rather than a neat filter. That's fine at
this scale (tens of incidents), and it's why the analysis cells below flatten to
a dataframe rather than asking Mongo to aggregate.

It's a genuine trade-off, and the right answer flips at scale. Worth knowing you
made it deliberately.

## 5. The five operations the app actually uses

Mongo has a big API surface; `app.py` uses five calls. Learn these and you can
read the whole sync layer.

| Call | Meaning |
|---|---|
| `find(filter, projection)` | read many documents |
| `find_one(filter)` | read one (or `None`) |
| `update_one(filter, changes, upsert=True)` | change one document, creating it if absent |
| `update_many(filter, changes)` | change every match |
| `delete_many(filter)` | remove every match |

A **filter** is a dict describing what to match — `{"incident_id": "INC-004"}`.
An empty filter `{}` means *everything*, which is powerful and dangerous.

**Changes** are never "here's the new document" — they're *operators* saying what
to modify, so you never accidentally wipe fields you didn't mention:

| Operator | Does |
|---|---|
| `$set` | set these fields (leave the rest alone) |
| `$setOnInsert` | set these **only if** this call creates the document |
| `$unset` | remove these fields |
| `$push` | append to an array |
| `$pull` | remove matching items from an array |

**Dotted paths reach into nesting:** `"by_document.R95RRQFZ.by_coder.coder1"` is
one field, four levels down. This is why the app can update one coder's work
without reading or rewriting the incident.

**Upsert** = update-or-insert. `update_one(..., upsert=True)` means "if no
incident has this id, create it". It's how a brand-new incident appears without a
separate insert path.

In [ ]:
# Read-only queries. Change the values and re-run — this is the way to learn.

# 1. Exact match, with a projection (1 = include this field, 0 = exclude)
print(coll.find_one({"incident_id": "INC-004"}, {"_id": 0, "incident_title": 1, "documents.url": 1}))

# 2. Which incident cites a given article? (this is what the documents.doc_id index is for)
print(coll.find_one({"documents.doc_id": "R95RRQFZ"}, {"_id": 0, "incident_id": 1}))
#    Note: matching INSIDE an array needs no special syntax — Mongo checks every element.

# 3. Count, and list every incident id
print(coll.count_documents({}), "incidents:", sorted(coll.distinct("incident_id")))

# 4. A comparison operator: incidents with more than one source article
multi = list(coll.find({"documents.1": {"$exists": True}}, {"_id": 0, "incident_id": 1}))
print("incidents with 2+ articles:", [m["incident_id"] for m in multi])
#    "documents.1" means "index 1 of the array" — if it exists, there are at least two.

# 5. Sorting and limiting
for d in coll.find({}, {"_id": 0, "incident_id": 1, "updated_at": 1}).sort("updated_at", -1).limit(3):
    print("most recently updated:", d)

## 6. How the app writes: `sync_to_mongo`, decoded

This is the hardest function in `app.py` ([lines 378–441](../app.py#L378-L441)),
and it's four writes in a row. Every save runs it.

**The problem it solves:** an article's incident is coder-editable. If someone
moves article `R95RRQFZ` from INC-004 to INC-009, the data must *move* — leaving a
copy behind would silently duplicate the research record. And it must move
**everyone's** coding, not just the person who dragged it.

```python
# 1. Gather every coder's existing reading of this article, wherever it currently lives
merged = {}
for inc in mongo_db.incidents.find({f"by_document.{key}": {"$exists": True}}, ...):
    merged.update(_by_coder(...))
merged[coder] = coding          # this save's contribution

# 2. Detach the article from any OTHER incident
update_many({"incident_id": {"$ne": inc_id}, ...},
            {"$unset": {f"by_document.{key}": ""}, "$pull": {"documents": {"doc_id": key}}})

# 3. Write it under its (new) incident — creating that incident if needed
update_one({"incident_id": inc_id},
           {"$setOnInsert": {...}, "$set": {f"by_document.{key}": {"by_coder": merged}},
            "$push": {"documents": doc_entry}}, upsert=True)

# 4. Delete any incident the move just emptied
delete_many({"documents": {"$size": 0}, ...})
```

Details worth noticing:

- **`$ne`** = not-equal. Step 2 says "every incident that is *not* the target".
- **`$pull` then `$push`** (the app pulls the doc entry first, then pushes) is a
  remove-then-add so the article can't end up listed twice.
- **Step 1 is the multi-coder fix.** Without it, step 2's `$unset` would delete the
  *other* coder's reading of an article you moved. Read the two together.
- **`$size: 0`** in step 4 matches incidents whose `documents` array is now empty,
  so a rename doesn't leave a hollow shell behind.

The reverse direction is `store_from_mongo(coder)` ([line 443](../app.py#L443)) —
used by **Pull from Mongo** to rebuild one coder's local JSON file.

## 7. Guardrails: the validator and the indexes

Mongo accepts any shape by default, which for a research dataset is a liability —
one typo'd field name and half your codings are invisible to a query. So
[`incidents_vocab.ensure_collection`](../incidents_vocab.py#L94) attaches a
**`$jsonSchema` validator** to the collection, and `app.py` calls it on every
startup (idempotent, so it just keeps the rules current).

It checks *structure* — `incident_id` is required and must be a string,
`documents` must be an array of objects with `doc_id`/`url`, each coder's coding
must have `quotes` as an array. It deliberately does **not** enforce the
vocabulary values, because those live under dynamic keys and are already
constrained by the UI (`vocab.json`).

`validationLevel: "moderate"` = apply the rules to new and updated documents, but
don't reject old ones that predate the rules. That's what let the per-coder shape
ship without a migration script.

**Indexes** are the other guardrail. An index makes lookups fast — and a *unique*
index makes a rule enforceable:

- `incident_id` **unique** → two documents can never claim the same incident.
- `documents.url`, `documents.doc_id` → "which incident cites this article?" stays
  fast as the collection grows.

In [ ]:
import json

info = db.command("listCollections", filter={"name": "incidents"})["cursor"]["firstBatch"][0]
rules = info.get("options", {}).get("validator", {}).get("$jsonSchema", {})
print("required:", rules.get("required"))
print("top-level properties:", sorted(rules.get("properties", {})))
print("\nvalidationLevel:", info.get("options", {}).get("validationLevel"))

print("\nindexes:")
for name, spec in coll.index_information().items():
    print(f"   {name}: keys={spec['key']} unique={spec.get('unique', False)}")

## 8. Getting it out: nested documents → a tidy table

Nesting is great for *storing* an incident and awkward for *analysing* one. For
analysis you want one row per observation — here, one row per
**incident × article × coder × role × value**. That's the input an intercoder
agreement statistic (Krippendorff's α, Cohen's κ) expects.

The loop below is the standard "flatten a document store" pattern: nested `for`
loops appending flat dicts, then one `DataFrame(rows)` at the end.

In [ ]:
import pandas as pd

LEGACY_CODER = "coder1"   # must match the FIRST name in the app's CODERS env var
ROLES = ["actor", "harm", "factor", "harmed_party"]


def by_coder(entry):
    """{coder: coding} for one by_document entry. Coding written before multi-coder
    support sat flat on the entry, and counts as the first coder's."""
    nested = entry.get("by_coder")
    if isinstance(nested, dict):
        return nested
    if {"fields", "quotes", "roles"} & set(entry):
        return {LEGACY_CODER: entry}
    return {}


rows = []
for inc in coll.find():
    titles = {d.get("doc_id"): d.get("title") for d in (inc.get("documents") or [])}
    for doc_key, entry in (inc.get("by_document") or {}).items():
        for coder, coding in by_coder(entry).items():
            for role in ROLES:
                for value in (coding.get("roles") or {}).get(role, []):
                    rows.append({"incident_id": inc.get("incident_id"),
                                 "incident_title": inc.get("incident_title"),
                                 "doc_key": doc_key, "doc_title": titles.get(doc_key),
                                 "coder": coder, "role": role, "value": value})

tidy = pd.DataFrame(rows)
print(f"{len(tidy)} coded values, {tidy.coder.nunique() if len(tidy) else 0} coder(s)")
tidy.head(15)

In [ ]:
# Where do coders agree? (Meaningful only once a second coder has coded something.)
if len(tidy) and tidy.coder.nunique() > 1:
    picked = (tidy.assign(picked=True)
                  .pivot_table(index=["incident_id", "role", "value"],
                               columns="coder", values="picked",
                               aggfunc="any", fill_value=False))
    both = picked.all(axis=1).sum()
    either = picked.any(axis=1).sum()
    print(f"values chosen by every coder: {both} / {either} chosen by anyone")
    print(f"raw agreement (Jaccard): {both / either:.0%}\n")
    display(picked[~picked.all(axis=1)].head(15))   # the disagreements
else:
    print("Only one coder has coded so far — nothing to compare yet.\n"
          "Code some documents as coder2 in the app, press Push to Mongo, and re-run.")

## 9. Which copy is the truth?

There are two stores and they can disagree, so it's worth being precise:

| | Local files | MongoDB Atlas |
|---|---|---|
| `annotations.<coder>.json` etc. | **source of truth** | mirror |
| Updated | on every keystroke (autosave) | on every save, plus **Push** |
| If the app can't reach Atlas | keeps working | falls behind, catch up with **Push** |
| Survives a Railway redeploy | ❌ (disk is wiped) | ✅ |

- **Push to Mongo** — send this coder's local work up. Never touches another
  coder's data.
- **Pull from Mongo** — bring this coder's work down; **Mongo wins** for any
  document in both. Local-only documents are kept, so a pull can't lose un-synced
  work, but local *edits* to a document Mongo also has will be overwritten.

Rule of thumb: **Push after a coding session, Pull before starting on another
machine.**

## 10. Safety, and the one dangerous cell

Mongo has no undo and no confirmation prompt. `delete_many({})` empties the
collection instantly and cheerfully reports success.

Habits worth forming:

1. **Run the filter as a `find` first.** See what matches *before* you delete it.
2. **Never type `{}` as a filter to a delete or an update** unless emptying the
   collection is genuinely the goal.
3. **Take a backup first** — the cell below writes every document to a JSON file.
   Your local `annotations.<coder>.json` files are also a backup: with them you can
   rebuild Atlas entirely by pressing **Push to Mongo**.
4. **Rotate the Atlas password** if it was ever pasted into a notebook or committed.

In [ ]:
# Safe: dump the whole collection to a timestamped file before doing anything risky.
import json
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
out = ROOT / f"backup-incidents-{stamp}.json"
out.write_text(json.dumps(list(coll.find()), indent=2, default=str))
print("wrote", out.name, f"({out.stat().st_size / 1024:.0f} KB)")

In [ ]:
# ⚠️ DESTRUCTIVE — deliberately left commented out. Read §10 before uncommenting.
#
# Step 1: ALWAYS look at what you're about to destroy.
# doomed = {"incident_id": "INC-900"}
# print(list(coll.find(doomed, {"_id": 0, "incident_id": 1, "incident_title": 1})))
#
# Step 2: only then, and only if that list is exactly what you meant:
# print(coll.delete_many(doomed).deleted_count, "deleted")
#
# To wipe everything and rebuild from your local files: coll.delete_many({}) then
# press "Push to Mongo" in the app, once per coder.
print("nothing was deleted — this cell only defines the recipe")

## 11. Try these

Best way to get fluent — each is one small edit to a cell above.

1. Find every incident whose title mentions AI:
   `coll.find({"incident_title": {"$regex": "AI", "$options": "i"}})`
2. Count how many articles are in the collection in total (hint: `$size` in an
   aggregation, or just `sum(len(i["documents"]) for i in coll.find())`).
3. List the incidents **coder2** has touched — scan `by_document.*.by_coder` for
   the key `"coder2"`.
4. Group the tidy table by `role` and `value` to find the most-coded harm:
   `tidy[tidy.role == "harm"].value.value_counts()`.
5. Break something on purpose in a scratch database: try inserting
   `{"incident_title": "no id"}` and watch the validator reject it because
   `incident_id` is required.

## Where to look next

- [`../app.py`](../app.py) lines 378–510 — every Mongo call the app makes.
- [`../incidents_vocab.py`](../incidents_vocab.py) — the validator and indexes.
- [`flask_guide.md`](flask_guide.md) — how the web app around it is built.
- [`../mongo_connect.ipynb`](../mongo_connect.ipynb) — your scratch notebook for
  quick looks at the data.